In [42]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [43]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [44]:
import json

from prompt_toolkit import prompt
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "python/json/regex"
        "solution_criteria": "Key criteria for evalution the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 10 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [45]:
dataset = generate_dataset()

with open ("dataset.json", "w") as f:
 json.dump(dataset, f, indent=4)


In [46]:
def run_prompt(test_case):
 """"Merges the prompt and test case input, then ruturn the result"""
 prompt = f""""
 Please soleve the following task:
{test_case['task']}
"""
 messages = []
 add_user_message(messages, prompt)
 output = chat(messages)
 return output

In [47]:
def grade_by_model(test_case, output):
     eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

citeria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>


Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
     
     message = []
     add_user_message(message, eval_prompt)
     add_assistant_message(message, "```json")
     eval_text = chat(message, stop_sequences=["```"])
     return json.loads(eval_text)

In [48]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    # TODO: - Grade
    model_grade = grade_by_model(test_case, output)
    return {
        "output": output,
        "test_case": test_case,
        "score": model_grade["score"],
        "reasoning": model_grade["reasoning"]
    }

In [49]:
from statistics import mean
def run_eval(test_case):
 """"Loads the dataset and calls run_test_case with each case"""
 results = []
 for test_case in dataset:
  result = run_test_case(test_case)
  results.append(result)

  average_score = mean([result["score"] for result in results])

  print(f"Average score so far: {average_score}")
  
 return results

In [50]:
with open ("dataset.json", "r") as f:
 dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Average score so far: 6
Average score so far: 5.5
Average score so far: 6
Average score so far: 6.25
Average score so far: 6.2
Average score so far: 6.333333333333333
Average score so far: 6.285714285714286
Average score so far: 6.25
Average score so far: 6.333333333333333
Average score so far: 6.4
[
  {
    "output": "# Extract AWS Region from S3 Bucket ARN\n\nHere are several solutions:\n\n## Solution 1: Using String Parsing (Simple approach)\n\n```python\ndef extract_region_from_s3_arn(arn):\n    \"\"\"\n    Extract region from S3 bucket ARN.\n    Note: S3 bucket ARNs don't contain region info directly.\n    This extracts region from the bucket name if it's included there.\n    \"\"\"\n    # S3 ARN format: arn:aws:s3:::bucket-name\n    bucket_name = arn.split(':::')[-1]\n    \n    # Common AWS regions in bucket names\n    regions = [\n        'us-east-1', 'us-east-2', 'us-west-1', 'us-west-2',\n        'eu-west-1', 'eu-central-1', 'ap-southeast-1', 'ap-northeast-1'\n    ]\n    \n   